# Scrape r/ChangeMyView (2013–2018) and draw the fixed annotation sample

This notebook rebuilds the study's data from the public [Arctic Shift](https://arctic-shift.photon-reddit.com) Reddit archive:

1. **Sections 1–7:** download every r/ChangeMyView post and comment from 2013–2018 (about 1.6 GB; resumable) and audit the download.
2. **Sections 8–10:** reconstruct delta events, sanitize the text (delta markers removed) and build the table of eligible direct responses.
3. **Section 11:** draw the fixed, behavior-stratified sample of 120 posts × 6 replies (seed 20260823) and record its SHA-256.
4. **Sections 13–19:** Monte Carlo power planning used to choose the sample size.
5. **Last cell:** write `data/annotation_input.csv` and `data/outcomes.csv`, the two files the rest of the repository uses.

The downloaded archive and intermediate files go to `data/archive/` (ignored by git). The outputs below are from the original run; the archive counts reported in the thesis (123,560 posts; 5,104,531 comments; 577,477 eligible direct responses) appear in them. Rerunning the download takes several hours.

In [ ]:
# ============================================================
# 1. Environment  (Colab-safe)
# ============================================================
# FIX D12: DuckDB goes to local ephemeral disk, not Drive.
# Do NOT upgrade pandas/numpy/pyarrow on Colab -- it breaks the runtime.

%pip install -q duckdb

import os, sys, gc, json, time, random, hashlib, threading, itertools
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import duckdb

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules

# Repository root: the first folder at or above the working directory that contains data/annotation_input.csv.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "annotation_input.csv").exists())
ROOT = REPO / "data" / "archive" / "cmv_pre2019_v3"     # raw + derived archive files (~1.6 GB, not in git)
if IN_COLAB:
    DB_DIR  = Path("/content/duckdb_scratch")         # local SSD on Colab
    TMP_DIR = Path("/content/duckdb_tmp")
else:
    DB_DIR  = ROOT / "scratch"
    TMP_DIR = ROOT / "scratch" / "tmp"
print("Archive folder:", ROOT)

print("duckdb", duckdb.__version__, "| pandas", pd.__version__)

duckdb 1.3.2 | pandas 2.2.3


In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

SUBREDDIT       = "changemyview"
SCRAPER_VERSION = 3

START_UTC = int(datetime(2013, 1, 1, tzinfo=timezone.utc).timestamp())
END_UTC   = int(datetime(2019, 1, 1, tzinfo=timezone.utc).timestamp())
YEARS     = list(range(2013, 2019))

API_BASE       = "https://arctic-shift.photon-reddit.com/api"
PAGE_LIMIT     = "auto"
BOUNDARY_LIMIT = 100

# 1 = safest. 2 ~= halves wall-clock and has been fine in practice.
# Do not exceed 3 on a public API.
MAX_WORKERS = 2

REQUEST_DELAY          = 0.25
REQUEST_TIMEOUT        = 120
MAX_CONSECUTIVE_ERRORS = 20
RATE_REMAINING_FLOOR   = 5
MAX_PAGES_PER_MONTH    = 4000
MAX_NO_NEW_PAGES       = 3

USER_AGENT = ("CMV-academic-research-scraper/3.0 "
              "(historical public Reddit archive; respectful Arctic Shift client)")

RAW_POSTS_DIR    = ROOT / "raw" / "posts"
RAW_COMMENTS_DIR = ROOT / "raw" / "comments"
DERIVED_DIR      = ROOT / "derived"
MANIFEST_PATH    = ROOT / "manifest.json"
DB_PATH          = DB_DIR / "cmv_reconstruction.duckdb"

for p in [ROOT, RAW_POSTS_DIR, RAW_COMMENTS_DIR, DERIVED_DIR, DB_DIR, TMP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

POST_FIELDS = ["id","author","subreddit","title","selftext","score",
               "created_utc","num_comments","url","over_18",
               "link_flair_text","author_flair_text","retrieved_on"]

COMMENT_FIELDS = ["id","author","subreddit","body","score","created_utc",
                  "link_id","parent_id","distinguished","author_flair_text",
                  "retrieved_on"]

# ---- FIX D2: delta detection is configurable and audited in Cell 8 ----
DELTA_BOT_AUTHORS = ["deltabot"]          # extended after the Cell 8 audit
DELTA_CONFIRM_RE  = 'delta[s]? awarded to|confirmed:?[[:space:]]*[0-9]+[[:space:]]+delta'
DELTA_REJECT_RE   = 'rejected|cannot award|can.t award|only award|failed'

# ---- FIX D7: strict stratum allocation, exact by construction ----
HUMAN_N_OPS               = 120
HUMAN_N_DELTA_POSITIVE_OPS = 100
HUMAN_N_NO_DELTA_OPS       = HUMAN_N_OPS - HUMAN_N_DELTA_POSITIVE_OPS
HUMAN_N_RESPONSES_PER_OP   = 6
DELTA_OP_ALLOC    = {"D": 1, "R": 2, "N": 3}   # -> 100 D, 200 R, 300 N
NO_DELTA_OP_ALLOC = {"D": 0, "R": 2, "N": 4}   # ->        40 R,  80 N
HUMAN_SAMPLE_SEED = 20260823

print("Root:", ROOT)
print("DuckDB:", DB_PATH, "(ephemeral, regenerable)")
print("Workers:", MAX_WORKERS)
print("Expected Drive footprint: ~1.6 GB")

Root: data/archive/cmv_pre2019_v3
DuckDB: /content/duckdb_scratch/cmv_reconstruction.duckdb (ephemeral, regenerable)
Workers: 2
Expected Drive footprint: ~1.6 GB


In [ ]:
# ============================================================
# 3. HTTP, checkpoint, and date helpers
# ============================================================

_print_lock    = threading.Lock()
_manifest_lock = threading.Lock()

def tprint(*a, **k):
    with _print_lock:
        print(*a, **k)

def load_manifest():
    if MANIFEST_PATH.exists():
        with open(MANIFEST_PATH) as f:
            return json.load(f)
    return {"version": SCRAPER_VERSION, "months": {}, "created_utc": int(time.time())}

MANIFEST = load_manifest()

def save_manifest():
    tmp = MANIFEST_PATH.with_suffix(".json.tmp")
    with open(tmp, "w") as f:
        json.dump(MANIFEST, f, indent=2, sort_keys=True)
    os.replace(tmp, MANIFEST_PATH)

def month_bounds(y, m):
    s = datetime(y, m, 1, tzinfo=timezone.utc)
    e = datetime(y + 1, 1, 1, tzinfo=timezone.utc) if m == 12 \
        else datetime(y, m + 1, 1, tzinfo=timezone.utc)
    return int(s.timestamp()), int(e.timestamp())

def month_key(entity, y, m):  return f"{entity}:{y}-{m:02d}"
def month_path(entity, y, m):
    base = RAW_POSTS_DIR if entity == "posts" else RAW_COMMENTS_DIR
    return base / f"{SUBREDDIT}_{entity}_{y}-{m:02d}.parquet"

def make_session():
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT})
    retry = Retry(total=8, connect=8, read=8, status=8, backoff_factor=1.0,
                  status_forcelist=[500,502,503,504], allowed_methods=["GET"],
                  raise_on_status=False)
    ad = HTTPAdapter(max_retries=retry,
                     pool_connections=max(2, MAX_WORKERS + 1),
                     pool_maxsize=max(2, MAX_WORKERS + 1))
    s.mount("https://", ad)
    return s

def _rate_wait_seconds(resp):
    for k in ("X-RateLimit-Reset", "x-ratelimit-reset"):
        v = resp.headers.get(k)
        if v is not None:
            try: return max(1.0, float(v))
            except Exception: pass
    return 20.0

def api_get(session, endpoint, params, label="request"):
    errors = 0
    while True:
        try:
            resp = session.get(f"{API_BASE}/{endpoint}", params=params,
                               timeout=REQUEST_TIMEOUT)
            if resp.status_code == 429:
                w = _rate_wait_seconds(resp)
                tprint(f"⏳ {label}: 429; sleeping {w:.1f}s")
                time.sleep(w); continue
            resp.raise_for_status()
            payload = resp.json()
            if not isinstance(payload, dict):
                raise RuntimeError(f"{label}: non-dict JSON")
            rem = (resp.headers.get("X-RateLimit-Remaining")
                   or resp.headers.get("x-ratelimit-remaining"))
            if rem is not None:
                try:
                    if float(rem) <= RATE_REMAINING_FLOOR:
                        w = _rate_wait_seconds(resp)
                        tprint(f"⏳ {label}: quota low ({rem}); sleeping {w:.1f}s")
                        time.sleep(w)
                except Exception: pass
            return payload
        except Exception as exc:
            errors += 1
            if errors > MAX_CONSECUTIVE_ERRORS:
                raise RuntimeError(f"{label}: failed after {MAX_CONSECUTIVE_ERRORS} "
                                   f"retries. Last: {exc}") from exc
            w = min(90.0, (2 ** min(errors, 6)) + random.random())
            tprint(f"⚠️  {label}: error {errors}/{MAX_CONSECUTIVE_ERRORS}; "
                   f"sleeping {w:.1f}s: {exc}")
            time.sleep(w)

def atomic_parquet_write(df, path):
    tmp = Path(str(path) + ".tmp")
    df.to_parquet(tmp, index=False, compression="zstd")
    os.replace(tmp, path)

def parquet_row_count(path):
    try: return int(pq.ParquetFile(path).metadata.num_rows)
    except Exception: return None

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

print("Helpers ready.")

Helpers ready.


In [ ]:
# ============================================================
# 4. Detect `after` boundary semantics
# ============================================================

def detect_after_semantics():
    s = make_session()
    ps = int(datetime(2013, 2, 1, tzinfo=timezone.utc).timestamp())
    pe = int(datetime(2013, 3, 1, tzinfo=timezone.utc).timestamp())

    d = (api_get(s, "posts/search",
                 {"subreddit": SUBREDDIT, "after": ps, "before": pe,
                  "sort": "asc", "limit": 10, "fields": "id,created_utc"},
                 "probe 1").get("data") or [])
    if not d:
        raise RuntimeError("Probe returned no Feb-2013 CMV posts.")
    t = int(d[-1]["created_utc"])

    d2 = (api_get(s, "posts/search",
                  {"subreddit": SUBREDDIT, "after": t,
                   "before": min(t + 86400, pe), "sort": "asc",
                   "limit": 100, "fields": "id,created_utc"},
                  "probe 2").get("data") or [])
    same = [x for x in d2 if int(x.get("created_utc", -1)) == t]
    inclusive = len(same) > 0

    print("Probe ts:", t, datetime.fromtimestamp(t, tz=timezone.utc))
    print("Detected `after` semantics:", "INCLUSIVE" if inclusive else "STRICT")
    return inclusive

AFTER_IS_INCLUSIVE = detect_after_semantics()

def after_to_start_at(ts):
    """First returned record should have created_utc >= ts."""
    return ts if AFTER_IS_INCLUSIVE else ts - 1

def after_to_step_past(ts):
    """First returned record should have created_utc > ts."""
    return ts + 1 if AFTER_IS_INCLUSIVE else ts

print("AFTER_IS_INCLUSIVE =", AFTER_IS_INCLUSIVE)

Probe ts: 1360189736 2013-02-06 22:28:56+00:00
Detected `after` semantics: STRICT
AFTER_IS_INCLUSIVE = False


In [ ]:
# ============================================================
# 5. Monthly scraper -- rewind-by-one-second cursor
# ============================================================
# FIX (performance): the old version issued an exact-second query after EVERY
# page (~16k extra requests). Rewinding the cursor to the page's final second
# makes the next page re-collect that second in full; dedup by ID absorbs the
# overlap. The exact-second query is now only needed when an entire page
# collapses to one timestamp (in practice: the month's tail).

def fetch_exact_second(session, entity, ts, fields, key):
    payload = api_get(session, f"{entity}/search",
                      {"subreddit": SUBREDDIT,
                       "after": after_to_start_at(ts), "before": ts + 1,
                       "sort": "asc", "limit": BOUNDARY_LIMIT,
                       "fields": ",".join(fields)},
                      f"{key} exact-second {ts}")
    raw = payload.get("data") or []
    exact = []
    for it in raw:
        try:
            if int(it.get("created_utc")) == ts:
                exact.append(it)
        except Exception:
            continue
    if len(exact) >= BOUNDARY_LIMIT:
        raise RuntimeError(
            f"{key}: >={BOUNDARY_LIMIT} {entity} share timestamp {ts}. "
            "Cannot certify this second via the search endpoint.")
    return exact


def validate_month_df(df, start_utc, end_utc, key):
    if len(df) == 0:
        return
    if df["id"].isna().any():        raise AssertionError(f"{key}: null IDs")
    if df["id"].duplicated().any():  raise AssertionError(f"{key}: duplicate IDs")
    ts = pd.to_numeric(df["created_utc"], errors="coerce")
    if ts.isna().any():              raise AssertionError(f"{key}: bad created_utc")
    if ((ts < start_utc) | (ts >= end_utc)).any():
        raise AssertionError(f"{key}: rows outside requested month")
    if not ts.is_monotonic_increasing:
        raise AssertionError(f"{key}: non-monotonic timestamps")


def fetch_month(entity, year, month, force=False):
    assert entity in {"posts", "comments"}
    fields   = POST_FIELDS if entity == "posts" else COMMENT_FIELDS
    endpoint = f"{entity}/search"

    start_utc, end_utc = month_bounds(year, month)
    start_utc = max(start_utc, START_UTC)
    end_utc   = min(end_utc, END_UTC)
    if start_utc >= end_utc:
        return 0

    key, out_path = month_key(entity, year, month), month_path(entity, year, month)

    if not force and out_path.exists():
        n_disk = parquet_row_count(out_path)
        info   = MANIFEST.get("months", {}).get(key, {})
        if (info.get("status") == "complete"
                and info.get("scraper_version") == SCRAPER_VERSION
                and n_disk == info.get("rows")):
            tprint(f"⏭️  {key}: {n_disk:,} rows already complete")
            return int(n_disk)

    session      = make_session()
    after_value  = after_to_start_at(start_utc)
    seen_ids     = set()
    rows         = []
    n_pages = n_exact_calls = no_new_streak = 0
    terminated_clean = False
    t0 = last_log = time.time()

    def _absorb(items):
        added = 0
        for it in items:
            rid, created = it.get("id"), it.get("created_utc")
            if rid is None or created is None:
                continue
            try: created = int(created)
            except Exception: continue
            if not (start_utc <= created < end_utc):
                continue
            rid = str(rid)
            if rid in seen_ids:
                continue
            rec = {f: it.get(f) for f in fields}
            rec["id"], rec["created_utc"] = rid, created
            rows.append(rec); seen_ids.add(rid); added += 1
        return added

    while True:
        if n_pages >= MAX_PAGES_PER_MONTH:
            raise RuntimeError(f"{key}: exceeded {MAX_PAGES_PER_MONTH} pages.")

        payload = api_get(session, endpoint,
                          {"subreddit": SUBREDDIT, "after": after_value,
                           "before": end_utc, "sort": "asc",
                           "limit": PAGE_LIMIT, "fields": ",".join(fields)},
                          f"{key} page {n_pages + 1}")
        data = payload.get("data") or []
        if not data:
            terminated_clean = True
            break
        n_pages += 1

        page_ts = []
        for it in data:
            try: c = int(it.get("created_utc"))
            except Exception: continue
            if start_utc <= c < end_utc:
                page_ts.append(c)
        if not page_ts:
            terminated_clean = True
            break

        n_new = _absorb(data)
        page_min, page_max = min(page_ts), max(page_ts)
        no_new_streak = 0 if n_new else no_new_streak + 1

        if page_min == page_max:
            # Whole page is one second: cannot rewind (would loop). Certify it,
            # then step past. This is normally just the month's final second.
            n_new += _absorb(fetch_exact_second(session, entity, page_max,
                                                fields, key))
            n_exact_calls += 1
            after_value = after_to_step_past(page_max)
        elif no_new_streak >= 1:
            # Rewound into a fully-collected second. Certify, then step past.
            n_new += _absorb(fetch_exact_second(session, entity, page_max,
                                                fields, key))
            n_exact_calls += 1
            after_value = after_to_step_past(page_max)
        else:
            # Normal: rewind so the final (possibly truncated) second is redone.
            after_value = after_to_start_at(page_max)

        if no_new_streak >= MAX_NO_NEW_PAGES:
            raise RuntimeError(f"{key}: {MAX_NO_NEW_PAGES} consecutive pages "
                               f"with no new rows at after={after_value}.")

        if time.time() - last_log >= 30:
            tprint(f"   {key}: {len(rows):,} rows | {n_pages:,} pages | "
                   f"{n_exact_calls} exact-second calls | "
                   f"{(time.time()-t0)/60:.1f} min")
            last_log = time.time()

        time.sleep(REQUEST_DELAY)

    df = pd.DataFrame(rows, columns=fields)
    if len(df):
        df = (df.drop_duplicates(subset=["id"], keep="first")
                .sort_values(["created_utc", "id"], kind="mergesort")
                .reset_index(drop=True))

    validate_month_df(df, start_utc, end_utc, key)
    if not terminated_clean:
        raise RuntimeError(f"{key}: loop did not end on an empty response.")
    atomic_parquet_write(df, out_path)

    with _manifest_lock:
        MANIFEST.setdefault("months", {})[key] = {
            "status": "complete", "scraper_version": SCRAPER_VERSION,
            "rows": int(len(df)), "start_utc": int(start_utc),
            "end_utc": int(end_utc), "main_pages": int(n_pages),
            "exact_second_calls": int(n_exact_calls),
            "max_utc_collected": int(df["created_utc"].max()) if len(df) else None,
            "terminated_clean": True,
            "after_is_inclusive": bool(AFTER_IS_INCLUSIVE),
            "completed_utc": int(time.time()),
        }
        save_manifest()

    tprint(f"✅ {key}: {len(df):,} rows | {n_pages:,} pages | "
           f"{n_exact_calls} exact-second | {(time.time()-t0)/60:.1f} min")
    del df, rows, seen_ids
    gc.collect()
    return int(MANIFEST["months"][key]["rows"])


def scrape_entity(entity, workers=MAX_WORKERS, force=False):
    jobs, total = [(y, m) for y in YEARS for m in range(1, 13)], 0
    tprint("\n" + "=" * 78)
    tprint(f"SCRAPING {entity.upper()} -- {len(jobs)} monthly partitions")
    tprint("=" * 78)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(fetch_month, entity, y, m, force): (y, m) for y, m in jobs}
        for fut in as_completed(futs):
            y, m = futs[fut]
            try:
                total += fut.result()
            except Exception as exc:
                tprint(f"❌ {entity}:{y}-{m:02d} FAILED\n{exc}")
                for f in futs: f.cancel()
                raise
    tprint(f"\n✅ {entity}: {total:,} rows")
    return total

print("Scraper ready.")

Scraper ready.


In [ ]:
# ============================================================
# 6. FULL SCRAPE  (resumable -- just re-run after a disconnect)
# ============================================================

_ = fetch_month("posts", 2013, 2, force=True)      # smoke test
print("Smoke test OK.\n")

posts_total    = scrape_entity("posts")
comments_total = scrape_entity("comments")

print("\n" + "=" * 78)
print(f"Posts:    {posts_total:,}")
print(f"Comments: {comments_total:,}")
print("Manifest:", MANIFEST_PATH)

✅ posts:2013-02: 174 rows | 2 pages | 1 exact-second | 0.0 min
Smoke test OK.


SCRAPING POSTS -- 72 monthly partitions
⏭️  posts:2013-01: 31 rows already complete
⏭️  posts:2013-02: 174 rows already complete
⏭️  posts:2013-04: 915 rows already complete
⏭️  posts:2013-03: 415 rows already complete
⏭️  posts:2013-05: 2,272 rows already complete
⏭️  posts:2013-06: 2,028 rows already complete
⏭️  posts:2013-08: 2,277 rows already complete
⏭️  posts:2013-07: 1,715 rows already complete
⏭️  posts:2013-09: 1,901 rows already complete
⏭️  posts:2013-10: 1,701 rows already complete
⏭️  posts:2013-11: 1,793 rows already complete
⏭️  posts:2013-12: 1,730 rows already complete
⏭️  posts:2014-01: 1,956 rows already complete
⏭️  posts:2014-02: 1,925 rows already complete
⏭️  posts:2014-03: 1,984 rows already complete
⏭️  posts:2014-04: 1,992 rows already complete
⏭️  posts:2014-05: 1,852 rows already complete
⏭️  posts:2014-06: 1,682 rows already complete
⏭️  posts:2014-08: 1,586 rows already compl

In [ ]:
# ============================================================
# 7. Raw audit  (FIX D4: the time series is now actually compared)
# ============================================================

def audit_raw_entity(entity):
    fails, rows = [], []
    for y in YEARS:
        for m in range(1, 13):
            key, path = month_key(entity, y, m), month_path(entity, y, m)
            s_utc, e_utc = month_bounds(y, m)
            if not path.exists():
                fails.append((key, "missing")); continue
            df = pd.read_parquet(path, columns=["id", "created_utc"])
            probs = []
            if df["id"].duplicated().any(): probs.append("dupe IDs")
            ts = pd.to_numeric(df["created_utc"], errors="coerce")
            if ts.isna().any(): probs.append("bad ts")
            if len(df) and ((ts < s_utc) | (ts >= e_utc)).any(): probs.append("out of month")
            if len(df) and not ts.is_monotonic_increasing: probs.append("non-monotonic")
            info = MANIFEST.get("months", {}).get(key, {})
            if int(info.get("rows", -1)) != len(df): probs.append("manifest mismatch")
            if not info.get("terminated_clean"): probs.append("unclean termination")
            # NEW: coverage -- did the partition reach the end of its month?
            if len(df) and (e_utc - int(ts.max())) > 21600:
                probs.append(f"tail gap {(e_utc-int(ts.max()))/3600:.1f}h")
            rows.append({"month": f"{y}-{m:02d}", "rows": len(df),
                         "status": "OK" if not probs else "; ".join(probs)})
            if probs: fails.append((key, probs))
    res = pd.DataFrame(rows)
    print(entity.upper(), "-", f"{res['rows'].sum():,} rows")
    display(res[res["status"] != "OK"] if (res["status"] != "OK").any()
            else res.head(3))
    if fails:
        print("⚠️  Issues (tail gaps in low-traffic 2013 months can be genuine):")
        for f in fails[:20]: print("  ", f)
    return res

posts_audit    = audit_raw_entity("posts")
comments_audit = audit_raw_entity("comments")

# ---- external completeness check, now with the comparison ----
def compare_time_series(kind, audit_df):
    s = make_session()
    payload = api_get(s, "time_series",
                      {"key": f"r/{SUBREDDIT}/{kind}/count", "precision": "month",
                       "after": START_UTC, "before": END_UTC}, f"ts {kind}")
    with open(DERIVED_DIR / f"arctic_shift_time_series_{kind}.json", "w") as f:
        json.dump(payload, f, indent=2)

    recs = payload.get("data") or payload.get("result") or []
    ext = {}
    for r in recs:
        if not isinstance(r, dict): continue
        k = r.get("key") or r.get("date") or r.get("time") or r.get("timestamp")
        v = r.get("value") or r.get("count") or r.get("doc_count")
        if k is None or v is None: continue
        try:
            ts = int(k)
            k = datetime.fromtimestamp(ts / (1000 if ts > 10**11 else 1),
                                       tz=timezone.utc).strftime("%Y-%m")
        except (TypeError, ValueError):
            k = str(k)[:7]
        ext[k] = ext.get(k, 0) + int(v)

    if not ext:
        print(f"\n{kind}: could not parse the time series. "
              "Inspect the saved JSON and map it manually.")
        return None

    cmp = audit_df.assign(external=audit_df["month"].map(ext))
    cmp["ratio"] = cmp["rows"] / cmp["external"]
    print(f"\n{kind}: scraped/external ratio -- "
          f"min {cmp['ratio'].min():.3f}, median {cmp['ratio'].median():.3f}, "
          f"max {cmp['ratio'].max():.3f}")
    bad = cmp[(cmp["ratio"] < 0.97) | (cmp["ratio"] > 1.03)]
    if len(bad):
        print("Months outside ±3% (Arctic Shift notes the series is approximate):")
        display(bad)
    else:
        print("All months within ±3%. ✅")
    return cmp

try:
    ts_posts    = compare_time_series("posts", posts_audit)
    ts_comments = compare_time_series("comments", comments_audit)
except Exception as exc:
    print("Time-series check unavailable:", exc)

POSTS - 123,560 rows


     month  rows status
0  2013-01    31     OK
1  2013-02   174     OK
2  2013-03   415     OK


COMMENTS - 5,104,531 rows


     month   rows status
0  2013-01    343     OK
1  2013-02   5730     OK
2  2013-03  14095     OK



posts: scraped/external ratio -- min 0.998, median 1.000, max 1.000
All months within ±3%. ✅

comments: scraped/external ratio -- min 1.000, median 1.000, max 1.000
All months within ±3%. ✅


In [ ]:
# ============================================================
# 2b. CONFIG OVERRIDE -- run after the diagnostic, before Cell 8
# ============================================================
# Cell 8 audit found an early-era DeltaBot variant: "Confirmed - N delta
# awarded to /u/X" (hyphen). The colon-only pattern missed ~2,530 events
# (~5% of deltas), concentrated in 2013-14. Accept both separators.

DELTA_BOT_AUTHORS = ["deltabot"]
DELTA_CONFIRM_RE  = '^[[:space:]]*confirmed[[:space:]]*(:|-)[[:space:]]*[0-9]+[[:space:]]+delta'
DELTA_REJECT_RE   = 'rejected|disallowed|cannot award|can.t award|only award|failed'

STRIP_EDIT_BLOCKS = False

print("Delta pattern:", DELTA_CONFIRM_RE)

Delta pattern: ^[[:space:]]*confirmed[[:space:]]*(:|-)[[:space:]]*[0-9]+[[:space:]]+delta


In [ ]:
# ============================================================
# 8. Delta-detection AUDIT (diagnostic -- changes nothing)
# ============================================================

con = duckdb.connect(str(DB_PATH))
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='6GB'")
con.execute(f"PRAGMA temp_directory='{TMP_DIR}'")

con.execute(f"""CREATE OR REPLACE VIEW raw_posts AS
  SELECT * FROM read_parquet('{RAW_POSTS_DIR}/*.parquet', union_by_name=true)""")
con.execute(f"""CREATE OR REPLACE VIEW raw_comments AS
  SELECT * FROM read_parquet('{RAW_COMMENTS_DIR}/*.parquet', union_by_name=true)""")

print("Posts:   ", f"{con.execute('SELECT COUNT(*) FROM raw_posts').fetchone()[0]:,}")
print("Comments:", f"{con.execute('SELECT COUNT(*) FROM raw_comments').fetchone()[0]:,}")

bots = "(" + ",".join(f"'{b}'" for b in DELTA_BOT_AUTHORS) + ")"

print("\n--- 8a. Any OTHER bot emitting delta-award language? ---")
display(con.execute("""
SELECT author, COUNT(*) n FROM raw_comments
WHERE regexp_matches(lower(coalesce(body,'')), 'delta[s]? awarded to')
GROUP BY 1 ORDER BY n DESC LIMIT 10""").df())

print("--- 8b. Coverage of the anchored pattern over DeltaBot comments ---")
display(con.execute(f"""
SELECT COUNT(*) all_bot_comments,
  SUM(regexp_matches(lower(coalesce(body,'')),'{DELTA_CONFIRM_RE}')::INT) matched,
  ROUND(100.0*SUM(regexp_matches(lower(coalesce(body,'')),'{DELTA_CONFIRM_RE}')::INT)
        /COUNT(*), 1) pct_matched
FROM raw_comments WHERE lower(trim(coalesce(author,''))) IN {bots}""").df())

print("--- 8c. Bot bodies the pattern MISSES -- read these ---")
display(con.execute(f"""
SELECT left(regexp_replace(coalesce(body,''),'[\\n\\r]+',' ','g'),95) snippet,
       COUNT(*) n
FROM raw_comments
WHERE lower(trim(coalesce(author,''))) IN {bots}
  AND NOT regexp_matches(lower(coalesce(body,'')),'{DELTA_CONFIRM_RE}')
GROUP BY 1 ORDER BY n DESC LIMIT 25""").df())

print("--- 8d. Confirmations per year (a cliff = format drift) ---")
display(con.execute(f"""
SELECT year(to_timestamp(created_utc)) yr, COUNT(*) confirmations
FROM raw_comments
WHERE lower(trim(coalesce(author,''))) IN {bots}
  AND regexp_matches(lower(coalesce(body,'')),'{DELTA_CONFIRM_RE}')
  AND NOT regexp_matches(lower(coalesce(body,'')),'{DELTA_REJECT_RE}')
GROUP BY 1 ORDER BY 1""").df())

Posts:    123,560
Comments: 5,104,531

--- 8a. Any OTHER bot emitting delta-award language? ---


(table of accounts posting delta-award language omitted: account names are not released)


--- 8b. Coverage of the anchored pattern over DeltaBot comments ---


   all_bot_comments  matched  pct_matched
0             71173  51727.0         72.7


--- 8c. Bot bodies the pattern MISSES -- read these ---


                                              snippet     n
0   This delta has been rejected. The length of yo...  1939
1   This delta is currently disallowed as your com...  1483
2   This delta has been rejected. The length of yo...   515
3   This delta has been rejected. You can't award ...   466
4   This award is currently disallowed as your com...   345
5   This delta is currently disallowed as your com...   330
6   You cannot award OP a delta as the moderators ...   319
7   This delta has been rejected. You have 2 issue...    95
8   This delta has been rejected. You cannot award...    77
9   /u/[user] (OP) has awarded 1 delta in ...    48
10  The moderators have confirmed that this is eit...    42
11  This delta has been rejected. You can't award ...    34
12  /u/[user] (OP) has awarded 1 delta in ...    29
13  You cannot award OP a delta as the moderators ...    28
14  This delta has been rejected. You can't award ...    25
15  /u/[user] (OP) has awarded 1 delta in this ...    24

--- 8d. Confirmations per year (a cliff = format drift) ---


     yr  confirmations
0  2013           5936
1  2014           6093
2  2015           6214
3  2016           6709
4  2017          11964
5  2018          14810


In [ ]:
print("--- Variant counts ---")
display(con.execute(f"""
SELECT
  SUM(regexp_matches(lower(body),'^[[:space:]]*confirmed:[[:space:]]*[0-9]+[[:space:]]+delta')::INT) colon,
  SUM(regexp_matches(lower(body),'^[[:space:]]*confirmed[[:space:]]*-[[:space:]]*[0-9]+[[:space:]]+delta')::INT) hyphen,
  SUM(regexp_matches(lower(body),'has awarded [0-9]+ delta')::INT) op_has_awarded
FROM raw_comments WHERE lower(trim(coalesce(author,''))) IN {bots}""").df())

print("--- Hyphen variant by year (expect early-era concentration) ---")
display(con.execute(f"""
SELECT year(to_timestamp(created_utc)) yr, COUNT(*) n
FROM raw_comments WHERE lower(trim(coalesce(author,''))) IN {bots}
  AND regexp_matches(lower(body),'^[[:space:]]*confirmed[[:space:]]*-[[:space:]]*[0-9]+[[:space:]]+delta')
GROUP BY 1 ORDER BY 1""").df())

print("--- Does 'has awarded' reply to a COMMENT (t1_) or the POST (t3_)? ---")
display(con.execute(f"""
SELECT left(CAST(parent_id AS VARCHAR),3) parent_type, COUNT(*) n
FROM raw_comments WHERE lower(trim(coalesce(author,''))) IN {bots}
  AND regexp_matches(lower(body),'has awarded [0-9]+ delta')
GROUP BY 1 ORDER BY n DESC""").df())

--- Variant counts ---


     colon  hyphen  op_has_awarded
0  49228.0  2499.0         12303.0


--- Hyphen variant by year (expect early-era concentration) ---


     yr     n
0  2013  2499


--- Does 'has awarded' reply to a COMMENT (t1_) or the POST (t3_)? ---


  parent_type      n
0         t3_  12303


In [ ]:
# ============================================================
# 9. Reconstruct delta events, sanitized text, direct responses
# ============================================================

# ---- 9A. Delta events (excludes '[deleted]' == '[deleted]' self-match)
con.execute(f"""
CREATE OR REPLACE TABLE delta_events AS
WITH c AS (
  SELECT *, regexp_replace(CAST(link_id AS VARCHAR),'^t3_','')   AS post_id,
            regexp_replace(CAST(parent_id AS VARCHAR),'^t1_','') AS parent_comment_id
  FROM raw_comments),
bot AS (
  SELECT id AS deltabot_comment_id, post_id, created_utc AS deltabot_created_utc,
         body AS deltabot_body, parent_comment_id AS award_comment_id
  FROM c
  WHERE lower(trim(coalesce(author,''))) IN {bots}
    AND regexp_matches(lower(coalesce(body,'')),'{DELTA_CONFIRM_RE}')
    AND NOT regexp_matches(lower(coalesce(body,'')),'{DELTA_REJECT_RE}')),
aw AS (
  SELECT b.*, a.author AS award_author, a.created_utc AS award_created_utc,
         regexp_replace(CAST(a.parent_id AS VARCHAR),'^t1_','') AS earning_comment_id
  FROM bot b LEFT JOIN c a ON a.id = b.award_comment_id),
ea AS (
  SELECT a.*, e.author AS earning_author
  FROM aw a LEFT JOIN c e ON e.id = a.earning_comment_id),
j AS (
  SELECT e.*, p.author AS post_author
  FROM ea e LEFT JOIN raw_posts p ON p.id = e.post_id)
SELECT *,
  (lower(trim(coalesce(award_author,''))) = lower(trim(coalesce(post_author,'')))
   AND lower(trim(coalesce(post_author,''))) NOT IN ('','[deleted]','[removed]')) AS op_awarded,
  (award_comment_id IS NOT NULL AND earning_comment_id IS NOT NULL
   AND award_author IS NOT NULL AND earning_author IS NOT NULL) AS tree_resolved
FROM j""")

con.execute(f"""COPY delta_events TO
 '{DERIVED_DIR}/changemyview_delta_events.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)""")
print("Delta events built.")

# ---- 9B. Attrition waterfall
print("\n" + "="*70); print("ATTRITION WATERFALL"); print("="*70)
display(con.execute("""
WITH c AS (SELECT *, regexp_replace(CAST(link_id AS VARCHAR),'^t3_','') post_id
           FROM raw_comments),
tl AS (SELECT DISTINCT post_id FROM c
       WHERE CAST(parent_id AS VARCHAR) = 't3_' || CAST(post_id AS VARCHAR)),
p1 AS (SELECT p.* FROM raw_posts p JOIN tl ON tl.post_id = p.id)
SELECT (SELECT COUNT(*) FROM raw_posts) s1_all_posts,
       (SELECT COUNT(*) FROM p1) s2_has_toplevel,
       (SELECT COUNT(*) FROM p1 WHERE lower(trim(coalesce(author,'')))
          NOT IN ('','[deleted]','automoderator')) s3_author_ok,
       (SELECT COUNT(*) FROM p1 WHERE lower(trim(coalesce(author,'')))
          NOT IN ('','[deleted]','automoderator')
          AND trim(coalesce(selftext,'')) <> ''
          AND lower(trim(coalesce(selftext,''))) NOT IN ('[deleted]','[removed]')
       ) s4_selftext_ok""").df())

# ---- 9C. Eligible direct responses + sanitization
# FIX: RE2 has no lookahead. Build the expression in Python instead of
# passing a no-op regex, and use [\s\S]* rather than the 's' flag.
DELTA_MK  = '(!delta|&#8710;|&#916;|&[Dd]elta;|∆|Δ)'
THANKS_LN = '[^\\n]*(thank[^\\n]{0,40}delta|delta[^\\n]{0,40}thank)[^\\n]*'
EDIT_FLAG = '(\\n|^)[ \\t]*(edit|eta|update)[a-z]*[ \\t]*[:\\-]'
ANY_CHARS = '[\\s\\S]*'

def clean_expr(col):
    e = f"regexp_replace({col}, '{THANKS_LN}', '', 'gi')"
    e = f"regexp_replace({e}, '{DELTA_MK}', '', 'g')"
    if STRIP_EDIT_BLOCKS:
        e = f"regexp_replace({e}, '{EDIT_FLAG}{ANY_CHARS}', '', 'i')"
    return e

OP_CLEAN, RESP_CLEAN = clean_expr("op_text_raw"), clean_expr("response_text_raw")

con.execute(f"""
CREATE OR REPLACE TABLE direct_base AS
WITH c AS (SELECT *, regexp_replace(CAST(link_id AS VARCHAR),'^t3_','') post_id
           FROM raw_comments),
j AS (
  SELECT c.id response_id, c.post_id, c.author response_author,
         c.body response_text_raw, c.score response_score,
         c.created_utc response_created_utc, c.retrieved_on response_retrieved_on,
         p.author op_author, p.title op_title, p.selftext op_text_raw,
         p.created_utc op_created_utc, p.score op_score,
         p.num_comments op_num_comments, p.link_flair_text op_link_flair_text, p.url op_url
  FROM c JOIN raw_posts p ON p.id = c.post_id
  WHERE CAST(c.parent_id AS VARCHAR) = 't3_' || CAST(c.post_id AS VARCHAR)),
f AS (
  SELECT * FROM j
  WHERE trim(coalesce(response_text_raw,'')) <> ''
    AND lower(trim(coalesce(response_text_raw,''))) NOT IN ('[deleted]','[removed]')
    AND lower(trim(coalesce(response_author,''))) NOT IN ('','[deleted]','automoderator','deltabot')
    AND lower(trim(coalesce(response_author,''))) <> lower(trim(coalesce(op_author,'')))
    AND trim(coalesce(op_title,'')) <> ''
    AND trim(coalesce(op_text_raw,'')) <> ''
    AND lower(trim(coalesce(op_text_raw,''))) NOT IN ('[deleted]','[removed]')
    AND lower(trim(coalesce(op_author,''))) NOT IN ('','[deleted]','automoderator')),
s AS (
  SELECT *,
    {OP_CLEAN}   AS op_text_clean,
    {RESP_CLEAN} AS response_text_clean,
    regexp_matches(op_text_raw,'{EDIT_FLAG}','i')       AS op_had_edit,
    regexp_matches(response_text_raw,'{EDIT_FLAG}','i') AS response_had_edit,
    regexp_matches(response_text_raw,'{DELTA_MK}','')   AS response_had_delta_marker,
    regexp_matches(lower(coalesce(response_text_raw,'')),'thank[^\\n]{{0,40}}delta','')
                                                        AS response_had_delta_thanks
  FROM f)
SELECT * FROM s
WHERE trim(coalesce(op_text_clean,'')) <> ''
  AND trim(coalesce(response_text_clean,'')) <> ''""")

print("\nSANITIZATION GUARD -- keep_ratio should be ~0.99+:")
display(con.execute("""
SELECT COUNT(*) n,
  ROUND(AVG(1.0*length(op_text_clean)/nullif(length(op_text_raw),0)),4) op_keep_ratio,
  ROUND(AVG(1.0*length(response_text_clean)/nullif(length(response_text_raw),0)),4) resp_keep_ratio,
  SUM((length(response_text_clean) < 0.8*length(response_text_raw))::INT) resp_lost_20pct,
  SUM(response_had_delta_marker::INT) had_marker,
  SUM(response_had_delta_thanks::INT) had_thanks,
  SUM(response_had_edit::INT) had_edit
FROM direct_base""").df())
print("If keep_ratio < 0.95 or resp_lost_20pct is large, STOP and inspect.")

# ---- 9D. Delta + reply summaries
con.execute("""
CREATE OR REPLACE TABLE direct_delta_summary AS
SELECT d.response_id, COUNT(e.deltabot_comment_id) n_confirmed_op_deltas,
       MIN(e.award_created_utc) first_op_delta_utc,
       any_value(e.award_comment_id) award_comment_id
FROM direct_base d
LEFT JOIN delta_events e ON e.earning_comment_id = d.response_id
  AND e.post_id = d.post_id AND e.op_awarded AND e.tree_resolved
GROUP BY d.response_id""")

# FIX: without `c.id IS NOT NULL`, the LEFT-JOIN null row scored as a reply,
# marking every no-reply response as replied-to.
con.execute("""
CREATE OR REPLACE TABLE op_reply_summary AS
SELECT d.response_id, COUNT(c.id) n_direct_op_replies,
  SUM(CASE WHEN c.id IS NOT NULL
             AND (x.award_comment_id IS NULL OR c.id <> x.award_comment_id)
           THEN 1 ELSE 0 END) n_op_replies_excl_award,
  MIN(c.created_utc) first_op_reply_utc
FROM direct_base d
LEFT JOIN direct_delta_summary x USING (response_id)
LEFT JOIN raw_comments c
  ON CAST(c.parent_id AS VARCHAR) = 't1_' || CAST(d.response_id AS VARCHAR)
 AND lower(trim(coalesce(c.author,''))) = lower(trim(coalesce(d.op_author,'')))
GROUP BY d.response_id""")

# ---- 9E. Final tables
con.execute("""
CREATE OR REPLACE TABLE direct_responses AS
WITH e AS (
  SELECT d.*,
    coalesce(r.n_direct_op_replies,0) > 0 AS op_reply,
    coalesce(r.n_op_replies_excl_award,0) > 0 AS op_reply_excl_award,
    coalesce(r.n_direct_op_replies,0) AS n_direct_op_replies,
    r.first_op_reply_utc,
    coalesce(x.n_confirmed_op_deltas,0) > 0 AS op_delta,
    coalesce(x.n_confirmed_op_deltas,0) AS n_confirmed_op_deltas,
    x.first_op_delta_utc,
    (d.response_created_utc - d.op_created_utc)/3600.0 AS hours_since_op
  FROM direct_base d
  LEFT JOIN op_reply_summary r USING (response_id)
  LEFT JOIN direct_delta_summary x USING (response_id)),
o AS (
  SELECT *,
    row_number() OVER (PARTITION BY post_id ORDER BY response_created_utc, response_id) response_order,
    count(*) OVER (PARTITION BY post_id) n_eligible_direct_responses,
    rank() OVER (PARTITION BY post_id ORDER BY response_score ASC NULLS FIRST) score_rank_min,
    count(*) OVER (PARTITION BY post_id, response_score) score_tie_n
  FROM e),
k AS (SELECT *, score_rank_min + (score_tie_n-1)/2.0 score_rank_average FROM o)
SELECT *,
  (score_rank_average - 0.5)/n_eligible_direct_responses AS community_score_percentile,
  CASE WHEN op_delta THEN 'D' WHEN op_reply THEN 'R' ELSE 'N' END AS behavior_state,
  CASE WHEN op_delta THEN 'D' WHEN op_reply_excl_award THEN 'R' ELSE 'N' END AS behavior_state_star
FROM k""")

con.execute(f"""COPY direct_responses TO
 '{DERIVED_DIR}/changemyview_direct_responses.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)""")

con.execute("""
CREATE OR REPLACE TABLE op_summary AS
SELECT post_id, any_value(op_author) op_author, any_value(op_title) op_title,
  any_value(op_created_utc) op_created_utc, COUNT(*) n_direct,
  SUM((behavior_state='D')::INT) n_delta,
  SUM((behavior_state='R')::INT) n_reply_no_delta,
  SUM((behavior_state='N')::INT) n_no_reply
FROM direct_responses GROUP BY post_id""")

con.execute(f"""COPY op_summary TO
 '{DERIVED_DIR}/changemyview_op_summary.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)""")
con.execute(f"""COPY op_summary TO
 '{DERIVED_DIR}/changemyview_op_summary.csv.gz' (HEADER, FORMAT CSV, COMPRESSION GZIP)""")
print("\nReconstruction complete.")

Delta events built.

ATTRITION WATERFALL


   s1_all_posts  s2_has_toplevel  s3_author_ok  s4_selftext_ok
0        123560            88113         64483           48782



SANITIZATION GUARD -- keep_ratio should be ~0.99+:


        n  op_keep_ratio  resp_keep_ratio  resp_lost_20pct  had_marker  \
0  577477         0.9997              1.0              5.0        84.0   

   had_thanks  had_edit  
0        11.0    9344.0  


If keep_ratio < 0.95 or resp_lost_20pct is large, STOP and inspect.



Reconstruction complete.


In [ ]:
# ============================================================
# 10. Quality checks
# ============================================================

print("DELTA EVENTS"); display(con.execute("""
SELECT COUNT(*) confirmed, SUM(tree_resolved::INT) resolved,
       SUM(op_awarded::INT) op_awarded, COUNT(DISTINCT post_id) posts
FROM delta_events""").df())

a = con.execute("""
SELECT COUNT(*) n_direct, COUNT(DISTINCT post_id) n_ops,
  SUM(op_reply::INT) n_reply, SUM(op_reply_excl_award::INT) n_reply_star,
  SUM(op_delta::INT) n_delta,
  SUM(CASE WHEN op_delta AND NOT op_reply THEN 1 ELSE 0 END) tautology_check,
  SUM(CASE WHEN op_delta AND NOT op_reply_excl_award THEN 1 ELSE 0 END)
    delta_with_no_other_reply
FROM direct_responses""").df()
display(a)

# FIX D11: the old assert was true by construction. Keep it as a sanity check,
# but the informative quantity is delta_with_no_other_reply.
assert int(a.loc[0,"tautology_check"]) == 0
print(f"\nDeltas whose ONLY OP reply was the award comment: "
      f"{int(a.loc[0,'delta_with_no_other_reply']):,} "
      f"({a.loc[0,'delta_with_no_other_reply']/a.loc[0,'n_delta']:.1%} of deltas)")
print("-> these are the cases the conditional D|R model cannot distinguish "
      "from the outcome itself. Report both D/R/N and D/R*/N.")

for label, q in [
    ("D/R/N", "SELECT behavior_state s, COUNT(*) n, COUNT(*)*1.0/SUM(COUNT(*)) OVER () p "
              "FROM direct_responses GROUP BY 1 ORDER BY 1"),
    ("D/R*/N", "SELECT behavior_state_star s, COUNT(*) n, COUNT(*)*1.0/SUM(COUNT(*)) OVER () p "
               "FROM direct_responses GROUP BY 1 ORDER BY 1"),
    ("Delta rate by year",
     "SELECT year(to_timestamp(op_created_utc)) yr, COUNT(*) n, "
     "AVG(op_delta::INT) delta_rate, AVG(op_reply::INT) reply_rate "
     "FROM direct_responses GROUP BY 1 ORDER BY 1"),
    ("Deltas awarded deeper in tree (NOT to a direct response)",
     "SELECT (SELECT SUM(op_awarded::INT) FROM delta_events WHERE tree_resolved) "
     "op_awarded_total, (SELECT SUM(op_delta::INT) FROM direct_responses) to_direct"),
    ("Sampling feasibility (strict allocation)",
     "SELECT COUNT(*) all_ops, SUM((n_direct>=6)::INT) ge6, "
     "SUM((n_direct>=6 AND n_delta>=1 AND n_reply_no_delta>=2 AND n_no_reply>=3)::INT) "
     "delta_pool, SUM((n_direct>=6 AND n_delta=0 AND n_reply_no_delta>=2 "
     "AND n_no_reply>=4)::INT) nodelta_pool FROM op_summary"),
]:
    print(f"\n{label}:"); display(con.execute(q).df())

DELTA EVENTS


   confirmed  resolved  op_awarded  posts
0      51726   51723.0     36959.0  26723


   n_direct  n_ops   n_reply  n_reply_star  n_delta  tautology_check  \
0    577477  48623  223574.0      206368.0  18712.0              0.0   

   delta_with_no_other_reply  
0                    17206.0  



Deltas whose ONLY OP reply was the award comment: 17,206 (92.0% of deltas)
-> these are the cases the conditional D|R model cannot distinguish from the outcome itself. Report both D/R/N and D/R*/N.

D/R/N:


   s       n         p
0  D   18712  0.032403
1  N  353903  0.612843
2  R  204862  0.354754



D/R*/N:


... (3 outputs left out)


Deltas awarded deeper in tree (NOT to a direct response):


   op_awarded_total  to_direct
0           36959.0    18712.0



Sampling feasibility (strict allocation):


   all_ops      ge6  delta_pool  nodelta_pool
0    48623  36710.0      7021.0       15616.0


In [ ]:
# ============================================================
# 11. Fixed human-validation sample
# ============================================================

rng = np.random.default_rng(HUMAN_SAMPLE_SEED)

def pool(where):
    return con.execute(f"SELECT post_id FROM op_summary WHERE {where} "
                       f"ORDER BY post_id").df()["post_id"].to_numpy()

pool_d = pool("n_direct>=6 AND n_delta>=1 AND n_reply_no_delta>=2 AND n_no_reply>=3")
pool_n = pool("n_direct>=6 AND n_delta=0 AND n_reply_no_delta>=2 AND n_no_reply>=4")
print(f"Delta-positive pool: {len(pool_d):,}  |  no-delta pool: {len(pool_n):,}")
assert len(pool_d) >= HUMAN_N_DELTA_POSITIVE_OPS, "delta pool too small"
assert len(pool_n) >= HUMAN_N_NO_DELTA_OPS,       "no-delta pool too small"

sel_d = np.sort(rng.choice(pool_d, HUMAN_N_DELTA_POSITIVE_OPS, replace=False))
sel_n = np.sort(rng.choice(pool_n, HUMAN_N_NO_DELTA_OPS,       replace=False))

con.register("sampled_ops", pd.DataFrame({
    "post_id": np.concatenate([sel_d, sel_n]),
    "op_stratum": ["delta_positive"]*len(sel_d) + ["no_delta"]*len(sel_n)}))

sub = con.execute("""
SELECT d.*, s.op_stratum FROM direct_responses d JOIN sampled_ops s USING (post_id)
ORDER BY d.post_id, d.response_id""").df()
print(f"Pulled {len(sub):,} rows / {sub.post_id.nunique()} OPs "
      f"({sub.memory_usage(deep=True).sum()/1e6:.0f} MB)")

pieces = []
for pid, g in sub.groupby("post_id", sort=True):
    alloc = DELTA_OP_ALLOC if g["op_stratum"].iloc[0] == "delta_positive" else NO_DELTA_OP_ALLOC
    for state, k in alloc.items():
        if k == 0: continue
        avail = g[g["behavior_state"] == state].sort_values("response_id")
        assert len(avail) >= k, f"{pid}: {state} has {len(avail)} < {k}"
        idx = np.sort(rng.choice(avail.index.to_numpy(), size=k, replace=False))
        pick = avail.loc[idx].copy()
        pick["inclusion_stratum"]        = state
        pick["n_available_in_stratum"]   = len(avail)
        pick["inclusion_prob_within_op"] = k / len(avail)   # needed for Study 4 weights
        pieces.append(pick)

human_sample = (pd.concat(pieces, ignore_index=True)
                  .sort_values(["post_id","response_id"]).reset_index(drop=True))
human_sample["human_sampling_seed"] = HUMAN_SAMPLE_SEED
human_sample["scraper_version"]     = SCRAPER_VERSION

assert len(human_sample) == HUMAN_N_OPS * HUMAN_N_RESPONSES_PER_OP
assert human_sample.groupby("post_id").size().eq(HUMAN_N_RESPONSES_PER_OP).all()

out = DERIVED_DIR / "human_validation_sample_v3.csv"
human_sample.to_csv(out, index=False)
digest = sha256_file(out)
(DERIVED_DIR / "human_validation_sample_v3.sha256").write_text(digest + "\n")

print(f"\nWrote: {out}\nSHA256: {digest}")
display(human_sample["behavior_state"].value_counts().rename_axis("state").to_frame("n"))
print("Expect exactly D=100, R=240, N=380")

leak = human_sample[human_sample["response_had_delta_marker"]
                    | human_sample["response_had_delta_thanks"]]
print(f"\nResponses with delta markers/thanks before sanitization: {len(leak)}/720")
print("Hand annotators + LLM the *_clean columns, NOT *_raw.")
print("\n>>> THE CSV + HASH ARE THE FIXED SAMPLE. Commit both. <<<")

Delta-positive pool: 7,021  |  no-delta pool: 15,616


Pulled 2,518 rows / 120 OPs (25 MB)

Wrote: data/archive/cmv_pre2019_v3/derived/human_validation_sample_v3.csv
SHA256: 289944870720a75f18b80c473185d76f223590c342f2e920d792044f835dd192


         n
state     
N      380
R      240
D      100


Expect exactly D=100, R=240, N=380

Responses with delta markers/thanks before sanitization: 0/720
Hand annotators + LLM the *_clean columns, NOT *_raw.

>>> THE CSV + HASH ARE THE FIXED SAMPLE. Commit both. <<<


In [ ]:
con = duckdb.connect(str(DB_PATH))   # if you already closed it
print("4-state dialogue-depth coding:")
display(con.execute("""
SELECT CASE
         WHEN op_delta AND op_reply_excl_award THEN 'D_engaged'
         WHEN op_delta                          THEN 'D_immediate'
         WHEN op_reply                          THEN 'R'
         ELSE 'N' END AS state,
       COUNT(*) n, ROUND(COUNT(*)*1.0/SUM(COUNT(*)) OVER (),5) p
FROM direct_responses GROUP BY 1 ORDER BY 1""").df())

4-state dialogue-depth coding:


         state       n        p
0    D_engaged    1506  0.00261
1  D_immediate   17206  0.02980
2            N  353903  0.61284
3            R  204862  0.35475


In [ ]:
# ============================================================
# 12. FIX D12: close DuckDB, verify Drive contents
# ============================================================
con.close()
print("DuckDB closed.")

tot = 0
for p in sorted(ROOT.rglob("*")):
    if p.is_file():
        mb = p.stat().st_size / 1024**2
        tot += mb
        if mb > 1 or "derived" in str(p) or p.name.endswith((".json", ".sha256")):
            print(f"{mb:9.1f} MB  {p.relative_to(ROOT)}")
print(f"\nTotal on Drive: {tot/1024:.2f} GB  (DuckDB is NOT here -- by design)")
print(f"DuckDB scratch: {DB_PATH} "
      f"({DB_PATH.stat().st_size/1024**3:.1f} GB, regenerable from Cell 8-9)")

DuckDB closed.
      0.0 MB  derived/arctic_shift_time_series_comments.json
      0.0 MB  derived/arctic_shift_time_series_posts.json
      2.4 MB  derived/changemyview_delta_events.parquet
    699.8 MB  derived/changemyview_direct_responses.parquet
      2.8 MB  derived/changemyview_op_summary.csv.gz
      2.4 MB  derived/changemyview_op_summary.parquet
      5.1 MB  derived/human_validation_sample_v3.csv
      0.0 MB  derived/human_validation_sample_v3.sha256
      0.0 MB  manifest.json
      1.4 MB  raw/comments/changemyview_comments_2013-02.parquet
      3.3 MB  raw/comments/changemyview_comments_2013-03.parquet
      8.7 MB  raw/comments/changemyview_comments_2013-04.parquet
     16.6 MB  raw/comments/changemyview_comments_2013-05.parquet
     17.5 MB  raw/comments/changemyview_comments_2013-06.parquet
     16.9 MB  raw/comments/changemyview_comments_2013-07.parquet
     23.8 MB  raw/comments/changemyview_comments_2013-08.parquet
     17.4 MB  raw/comments/changemyview_comments_20

In [ ]:
# ============================================================
# 13. MONTE CARLO POWER PLANNING — SETUP
# ============================================================
%pip install -q scipy

import itertools
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import t as student_t
from IPython.display import display

# ---- Corpus quantities from the CORRECTED reconstruction (Cell 10) ----
N_DIRECT, N_OPS = 577_477, 48_623
N_REPLY, N_DELTA = 223_574, 18_712
N_DELTA_IMMEDIATE = 17_206          # delta whose only OP reply was the award

P_DELTA   = N_DELTA / N_DIRECT
P_REPLY   = N_REPLY / N_DIRECT
P_D_GIV_R = N_DELTA / N_REPLY
P_R_GIV_ND = (N_REPLY - N_DELTA) / (N_DIRECT - N_DELTA)

print("Corrected CMV behaviour rates")
print(f"P(D)      = {P_DELTA:.4f}")
print(f"P(R)      = {P_REPLY:.4f}")
print(f"P(D|R)    = {P_D_GIV_R:.4f}")
print(f"P(R|~D)   = {P_R_GIV_ND:.4f}")
print(f"Deltas with NO prior OP exchange: {N_DELTA_IMMEDIATE/N_DELTA:.1%}")

# ---- Load the FIXED v3 validation sample (100 D / 240 R / 380 N) ----
try:
    mc_human_sample = human_sample.copy()
    print("\nUsing `human_sample` from memory.")
except NameError:
    mc_human_sample = pd.read_csv(DERIVED_DIR / "human_validation_sample_v3.csv")
    print("\nLoaded human_validation_sample_v3.csv")

op_sizes = mc_human_sample.groupby("post_id").size()
assert op_sizes.nunique() == 1 and int(op_sizes.iloc[0]) == 6
print("OPs:", mc_human_sample.post_id.nunique(), "| responses:", len(mc_human_sample))
display(mc_human_sample["behavior_state"].value_counts()
        .rename_axis("behavior_state").to_frame("n"))
assert (mc_human_sample["behavior_state"].value_counts()
        .reindex(["D","R","N"]).tolist() == [100,240,380]), "expected 100/240/380"

# ---- Monte Carlo settings ----
J_GRID   = [80, 100, 120, 150]
N_MC     = 2000          # thesis-grade; batched fit makes this cheap
N_BOOT_POWER = 1000
MC_SEED  = 20260824

N_RESPONSES_PER_OP, N_PAIRS_PER_OP = 6, 15
N_HUMAN_RATERS, N_LLM_REPEATS = 3, 3
BT_RIDGE = 0.10

PAIR_IDX = np.array(list(itertools.combinations(range(N_RESPONSES_PER_OP), 2)), dtype=int)
BT_X = np.zeros((len(PAIR_IDX), N_RESPONSES_PER_OP))
for r, (a, b) in enumerate(PAIR_IDX):
    BT_X[r, a], BT_X[r, b] = 1.0, -1.0
BT_I = np.eye(N_RESPONSES_PER_OP)

MC_DIR = DERIVED_DIR / "monte_carlo_power"
MC_DIR.mkdir(parents=True, exist_ok=True)
print("\nOutput:", MC_DIR, "| J grid:", J_GRID, "| N_MC:", N_MC)

Corrected CMV behaviour rates
P(D)      = 0.0324
P(R)      = 0.3872
P(D|R)    = 0.0837
P(R|~D)   = 0.3666
Deltas with NO prior OP exchange: 92.0%

Using `human_sample` from memory.
OPs: 120 | responses: 720


                  n
behavior_state     
N               380
R               240
D               100



Output: data/archive/cmv_pre2019_v3/derived/monte_carlo_power | J grid: [80, 100, 120, 150] | N_MC: 2000


In [ ]:
# ============================================================
# 14. STUDY 1 — BATCHED SIMULATION MACHINERY
# ============================================================

def bt_fit_batched(wins_a, k, ridge=BT_RIDGE, max_iter=50, tol=1e-8):
    """Regularized Bradley-Terry, fitted for B OPs simultaneously.
    wins_a: (B,15) counts preferring item a; k: judgments per pair."""
    B = wins_a.shape[0]
    theta = np.zeros((B, N_RESPONSES_PER_OP))
    I = BT_I[None, :, :]
    for _ in range(max_iter):
        p = expit(theta @ BT_X.T)                                  # (B,15)
        grad = (wins_a - k * p) @ BT_X - ridge * theta              # (B,6)
        w = k * p * (1.0 - p)                                       # (B,15)
        info = np.einsum('ri,br,rj->bij', BT_X, w, BT_X) + ridge * I
        step = np.linalg.solve(info, grad[:, :, None])[:, :, 0]
        new = theta + step
        new -= new.mean(axis=1, keepdims=True)
        if np.max(np.abs(new - theta)) < tol:
            return new
        theta = new
    return theta


def _bt_fit_scalar(wins_a, k, ridge=BT_RIDGE, max_iter=50, tol=1e-8):
    """Reference implementation for the self-test."""
    theta = np.zeros(N_RESPONSES_PER_OP)
    for _ in range(max_iter):
        p = expit(BT_X @ theta)
        grad = BT_X.T @ (wins_a - k * p) - ridge * theta
        w = k * p * (1 - p)
        info = BT_X.T @ (w[:, None] * BT_X) + ridge * BT_I
        new = theta + np.linalg.solve(info, grad)
        new -= new.mean()
        if np.max(np.abs(new - theta)) < tol:
            return new
        theta = new
    return theta

# ---- SELF-TEST: batched must equal scalar ----
_rng = np.random.default_rng(0)
_w = _rng.integers(0, 4, size=(7, 15))
_batched = bt_fit_batched(_w, 3)
_scalar = np.array([_bt_fit_scalar(_w[i], 3) for i in range(7)])
_err = np.max(np.abs(_batched - _scalar))
assert _err < 1e-9, f"batched BT disagrees with scalar (max err {_err:.2e})"
print(f"BT self-test passed (max abs diff {_err:.2e})")


def tau_b_batched(x, y):
    """Kendall tau-b over 15 pairs, for B OPs at once. x,y: (B,6) -> (B,)"""
    dx = x[:, PAIR_IDX[:, 0]] - x[:, PAIR_IDX[:, 1]]
    dy = y[:, PAIR_IDX[:, 0]] - y[:, PAIR_IDX[:, 1]]
    num = (np.sign(dx) * np.sign(dy)).sum(axis=1)
    den = np.sqrt((dx != 0).sum(axis=1) * (dy != 0).sum(axis=1))
    return np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan)


def correlation_matrix(rho_human, rho_match, rho_cross, rho_llm):
    """Latent corr for HC, HE, LC, LE.

    The matrix is symmetric under swapping C<->E, so positive-definiteness
    reduces to  (rho_match - rho_cross)^2 < (1-rho_human)(1-rho_llm).
    Strong human C/E coupling therefore CAPS attainable selective
    correspondence -- these are not independent knobs.
    """
    R = np.array([
        [1.0,       rho_human, rho_match, rho_cross],
        [rho_human, 1.0,       rho_cross, rho_match],
        [rho_match, rho_cross, 1.0,       rho_llm  ],
        [rho_cross, rho_match, rho_llm,   1.0      ]])
    ev = np.linalg.eigvalsh(R)
    if ev.min() <= 0:
        raise ValueError(f"latent matrix not PD; eigenvalues={ev}")
    return R


def krippendorff_alpha_binary(wins_a, k):
    """Approximate binary nominal alpha from simulated forced choices."""
    w = wins_a.ravel().astype(float)
    agree = (w * (w - 1) / 2 + (k - w) * (k - w - 1) / 2) / (k * (k - 1) / 2)
    A_o = agree.mean()
    p1 = w.mean() / k
    D_e = 2 * p1 * (1 - p1)
    return np.nan if D_e <= 0 else 1.0 - (1.0 - A_o) / D_e


def simulate_study1_dataset(J, sc, rng, want_alpha=False):
    """One Study 1 dataset of J OPs. Fully vectorized over OPs."""
    R = correlation_matrix(sc["rho_human"], sc["rho_match"],
                           sc["rho_cross"], sc["rho_llm"])
    latent = rng.multivariate_normal(np.zeros(4), R, size=(J, N_RESPONSES_PER_OP))
    latent -= latent.mean(axis=1, keepdims=True)          # within-OP centering

    fits, alphas = [], []
    cfg = [(0, N_HUMAN_RATERS, sc["human_choice_scale"]),   # HC
           (1, N_HUMAN_RATERS, sc["human_choice_scale"]),   # HE
           (2, N_LLM_REPEATS,  sc["llm_choice_scale"]),     # LC
           (3, N_LLM_REPEATS,  sc["llm_choice_scale"])]     # LE
    for col, k, scale in cfg:
        diff = latent[:, :, col] @ BT_X.T
        wins = rng.binomial(k, expit(scale * diff))
        fits.append(bt_fit_batched(wins, k))
        if want_alpha and col in (0, 1):
            alphas.append(krippendorff_alpha_binary(wins, k))
    HC, HE, LC, LE = fits

    dC = tau_b_batched(HC, LC) - tau_b_batched(HC, LE)
    dE = tau_b_batched(HE, LE) - tau_b_batched(HE, LC)
    gm = tau_b_batched(LC, LE) - tau_b_batched(HC, HE)
    return dC, dE, gm, (float(np.nanmean(alphas)) if alphas else np.nan)


def holm_rejections(p_values, alpha=0.05):
    p = np.asarray(p_values, float); m = len(p)
    rej = np.zeros(m, bool)
    for rank, idx in enumerate(np.argsort(p)):
        if p[idx] <= alpha / (m - rank):
            rej[idx] = True
        else:
            break
    return rej


def mean_pvalues(v):
    """Returns (one-sided p for mean>0, two-sided p)."""
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    n, mean, sd = len(v), v.mean(), v.std(ddof=1)
    if sd == 0:
        return (0.0, 0.0) if mean != 0 else (1.0, 1.0)
    t = mean / (sd / np.sqrt(n))
    return student_t.sf(t, df=n - 1), 2 * student_t.sf(abs(t), df=n - 1)


def run_study1_power(J_values, name, sc, n_mc=N_MC, seed=MC_SEED):
    rows = []
    for J in J_values:
        rng = np.random.default_rng(seed)      # common random numbers across J
        rec = []
        for i in range(n_mc):
            dC, dE, gm, al = simulate_study1_dataset(J, sc, rng,
                                                     want_alpha=(i < 50))
            p1 = [mean_pvalues(dC)[0], mean_pvalues(dE)[0], mean_pvalues(gm)[0]]
            p2 = [mean_pvalues(dC)[1], mean_pvalues(dE)[1], mean_pvalues(gm)[1]]
            r1, r2 = holm_rejections(p1), holm_rejections(p2)
            rec.append({"dC": np.nanmean(dC), "dE": np.nanmean(dE),
                        "gm": np.nanmean(gm), "alpha": al,
                        "rC": r1[0], "rE": r1[1], "rG": r1[2],
                        "rAll": r1.all(), "rG2": r2[2]})
        d = pd.DataFrame(rec)
        rows.append({"scenario": name, "J": J, "n_mc": n_mc,
                     "mean_delta_C": d.dC.mean(), "mean_delta_E": d.dE.mean(),
                     "mean_gamma": d.gm.mean(),
                     "implied_human_alpha": d.alpha.mean(),
                     "power_delta_C": d.rC.mean(), "power_delta_E": d.rE.mean(),
                     "power_gamma_1sided": d.rG.mean(),
                     "power_gamma_2sided": d.rG2.mean(),
                     "power_all_three": d.rAll.mean()})
        print(f"{name:15s} J={J:3d} | ΔC={d.rC.mean():.3f} ΔE={d.rE.mean():.3f} "
              f"Γ1={d.rG.mean():.3f} Γ2={d.rG2.mean():.3f} all={d.rAll.mean():.3f} "
              f"| mean Γ={d.gm.mean():+.3f} | α≈{d.alpha.mean():.2f}")
    return pd.DataFrame(rows)

print("Study 1 machinery ready.")

BT self-test passed (max abs diff 3.33e-16)
Study 1 machinery ready.


In [ ]:
# ============================================================
# 15. STUDY 1 — POWER CURVES (4 scenarios)
# ============================================================
STUDY1_SCENARIOS = {
    "conservative": dict(rho_human=0.40, rho_match=0.55, rho_cross=0.40,
                         rho_llm=0.50, human_choice_scale=1.00, llm_choice_scale=1.25),
    "planning":     dict(rho_human=0.35, rho_match=0.60, rho_cross=0.35,
                         rho_llm=0.55, human_choice_scale=1.20, llm_choice_scale=1.50),
    "favorable":    dict(rho_human=0.30, rho_match=0.65, rho_cross=0.30,
                         rho_llm=0.60, human_choice_scale=1.40, llm_choice_scale=1.70),
    # STRESS TEST: rho_human near Wachsmuth (2017) r(Cogency, Effectiveness)=.81.
    # Gamma estimates rho_llm - rho_human, so this makes Gamma NEGATIVE and the
    # pre-specified one-sided test (mean > 0) powerless. If Gamma < 0 here,
    # RQ1b must be restated as two-sided or as an equivalence test.
    "wachsmuth_like": dict(rho_human=0.78, rho_match=0.60, rho_cross=0.35,
                           rho_llm=0.55, human_choice_scale=1.20, llm_choice_scale=1.50),
}

res = []
for i, (nm, sc) in enumerate(STUDY1_SCENARIOS.items()):
    print("\n" + "=" * 78); print("SCENARIO:", nm.upper()); print("=" * 78)
    res.append(run_study1_power(J_GRID, nm, sc, N_MC, MC_SEED + i * 10_000))
study1_power = pd.concat(res, ignore_index=True)

print("\n" + "=" * 78)
print("GAMMA SIGN CHECK  (negative => RQ1b direction is backwards)")
print("=" * 78)
display(study1_power.pivot(index="J", columns="scenario", values="mean_gamma").round(3))
print("\nOne-sided Gamma power by scenario:")
display(study1_power.pivot(index="J", columns="scenario",
                           values="power_gamma_1sided").round(3))
print("\nImplied human Krippendorff alpha (calibration of human_choice_scale):")
display(study1_power.groupby("scenario")["implied_human_alpha"].mean().round(3))

display(study1_power)
study1_power.to_csv(MC_DIR / "study1_power_results.csv", index=False)
print("\nSaved study1_power_results.csv")


SCENARIO: CONSERVATIVE
conservative    J= 80 | ΔC=0.448 ΔE=0.444 Γ1=0.309 Γ2=0.199 all=0.127 | mean Γ=+0.074 | α≈0.27
conservative    J=100 | ΔC=0.545 ΔE=0.561 Γ1=0.376 Γ2=0.267 all=0.190 | mean Γ=+0.074 | α≈0.27
conservative    J=120 | ΔC=0.647 ΔE=0.670 Γ1=0.459 Γ2=0.320 all=0.289 | mean Γ=+0.074 | α≈0.27
conservative    J=150 | ΔC=0.762 ΔE=0.762 Γ1=0.569 Γ2=0.424 all=0.414 | mean Γ=+0.074 | α≈0.27

SCENARIO: PLANNING
planning        J= 80 | ΔC=0.956 ΔE=0.963 Γ1=0.843 Γ2=0.742 all=0.800 | mean Γ=+0.135 | α≈0.33
planning        J=100 | ΔC=0.984 ΔE=0.987 Γ1=0.919 Γ2=0.854 all=0.901 | mean Γ=+0.136 | α≈0.33
planning        J=120 | ΔC=0.997 ΔE=0.996 Γ1=0.952 Γ2=0.906 all=0.946 | mean Γ=+0.135 | α≈0.34
planning        J=150 | ΔC=1.000 ΔE=0.999 Γ1=0.984 Γ2=0.966 all=0.982 | mean Γ=+0.135 | α≈0.34

SCENARIO: FAVORABLE
favorable       J= 80 | ΔC=1.000 ΔE=1.000 Γ1=0.995 Γ2=0.989 all=0.995 | mean Γ=+0.200 | α≈0.39
favorable       J=100 | ΔC=1.000 ΔE=1.000 Γ1=0.999 Γ2=0.999 all=0.999 | mean Γ=+

scenario  conservative  favorable  planning  wachsmuth_like
J                                                          
80               0.074      0.200     0.135          -0.136
100              0.074      0.199     0.136          -0.133
120              0.074      0.201     0.135          -0.135
150              0.074      0.199     0.135          -0.135



One-sided Gamma power by scenario:


scenario  conservative  favorable  planning  wachsmuth_like
J                                                          
80               0.310      0.995     0.842             0.0
100              0.376      0.999     0.919             0.0
120              0.459      1.000     0.952             0.0
150              0.569      1.000     0.984             0.0



Implied human Krippendorff alpha (calibration of human_choice_scale):


scenario
conservative      0.274
favorable         0.389
planning          0.335
wachsmuth_like    0.334
Name: implied_human_alpha, dtype: float64


          scenario    J  n_mc  mean_delta_C  mean_delta_E  mean_gamma  \
0     conservative   80  2000      0.086431      0.086485    0.074486   
1     conservative  100  2000      0.084968      0.086277    0.074233   
2     conservative  120  2000      0.087107      0.087793    0.074013   
3     conservative  150  2000      0.086275      0.087387    0.074242   
4         planning   80  2000      0.153080      0.152541    0.134933   
5         planning  100  2000      0.152101      0.152982    0.135588   
6         planning  120  2000      0.153306      0.154024    0.135005   
7         planning  150  2000      0.153885      0.153451    0.135230   
8        favorable   80  2000      0.222612      0.223492    0.200174   
9        favorable  100  2000      0.223793      0.223618    0.199181   
10       favorable  120  2000      0.224991      0.224803    0.200891   
11       favorable  150  2000      0.224104      0.223904    0.199102   
12  wachsmuth_like   80  2000      0.154908      0.


Saved study1_power_results.csv


In [ ]:
# ============================================================
# 16. STUDY 3 — EMPIRICAL PAIR-INFORMATIVENESS TEMPLATES
# ============================================================
tpl = []
for pid, g in mc_human_sample.groupby("post_id", sort=True):
    g = g.sort_values("response_id").reset_index(drop=True)
    assert len(g) == 6
    D = g["behavior_state"].eq("D").to_numpy()
    U = pd.to_numeric(g["community_score_percentile"], errors="coerce").to_numpy()
    d_inf = int(sum(D[a] != D[b] for a, b in PAIR_IDX))
    u_inf = int(sum(np.isfinite(U[a]) and np.isfinite(U[b]) and U[a] != U[b]
                    for a, b in PAIR_IDX))
    tpl.append({"post_id": pid, "n_D_informative_pairs": d_inf,
                "n_U_informative_pairs": u_inf, "n_delta_responses": int(D.sum())})

study3_template_df = pd.DataFrame(tpl)
display(study3_template_df.describe().round(3))
print("D-informative pairs total:", study3_template_df.n_D_informative_pairs.sum())
print("U-informative pairs total:", study3_template_df.n_U_informative_pairs.sum())
print("\nNOTE: with strict 1-D-per-delta-OP allocation, D-informative pairs within an")
print("OP nearly all involve the SAME delta response -> wins are correlated, not iid.")
study3_template_df.to_csv(MC_DIR / "study3_behavioral_pair_templates.csv", index=False)

       n_D_informative_pairs  n_U_informative_pairs  n_delta_responses
count                120.000                120.000            120.000
mean                   4.167                 11.975              0.833
std                    1.871                  2.782              0.374
min                    0.000                  0.000              0.000
25%                    5.000                 11.000              1.000
50%                    5.000                 12.000              1.000
75%                    5.000                 14.000              1.000
max                    5.000                 15.000              1.000


D-informative pairs total: 500
U-informative pairs total: 1437

NOTE: with strict 1-D-per-delta-OP allocation, D-informative pairs within an
OP nearly all involve the SAME delta response -> wins are correlated, not iid.


In [ ]:
# ============================================================
# 17. STUDY 3 — MONTE CARLO POWER (corrected)
# ============================================================
# Two corrections vs. the earlier version, both of which LOWER power:
#  (a) within-OP win correlation: the D-informative pairs in an OP share one
#      delta response, so one Bernoulli per OP -- not one per pair.
#  (b) misclassification of the C/E-discordance label Z by majority-of-3
#      raters attenuates the true contrast toward zero by (1 - 2*z_error).

def study3_one_dataset(J, discordance_rate, pi_U, delta_pi, rng,
                       n_boot=N_BOOT_POWER, d_correlated=True, z_error=0.0):
    t = study3_template_df.iloc[rng.integers(0, len(study3_template_df), size=J)]
    nD = rng.binomial(t.n_D_informative_pairs.to_numpy(int), discordance_rate)
    nU = rng.binomial(t.n_U_informative_pairs.to_numpy(int), discordance_rate)

    att = 1.0 - 2.0 * z_error
    piD = 0.5 + att * ((pi_U + delta_pi) - 0.5)
    piU = 0.5 + att * (pi_U - 0.5)
    if not (0 <= piD <= 1 and 0 <= piU <= 1):
        raise ValueError("effective pi out of range")

    if d_correlated:
        winD = np.where(rng.random(J) < piD, nD, 0)   # one draw per OP
    else:
        winD = rng.binomial(nD, piD)
    winU = rng.binomial(nU, piU)

    tD, tU = nD.sum(), nU.sum()
    if tD == 0 or tU == 0:
        return {"delta_hat": np.nan, "reject": False,
                "n_D_discordant": tD, "n_U_discordant": tU}
    delta_hat = winD.sum() / tD - winU.sum() / tU

    bi = rng.integers(0, J, size=(n_boot, J))
    bDn, bUn = nD[bi].sum(1), nU[bi].sum(1)
    ok = (bDn > 0) & (bUn > 0)
    if ok.sum() < 50:
        return {"delta_hat": delta_hat, "reject": False,
                "n_D_discordant": tD, "n_U_discordant": tU}
    bd = winD[bi].sum(1)[ok] / bDn[ok] - winU[bi].sum(1)[ok] / bUn[ok]
    return {"delta_hat": delta_hat, "reject": bool(np.quantile(bd, 0.025) > 0),
            "n_D_discordant": tD, "n_U_discordant": tU}


def run_study3_power(J_values, discordance_rate, pi_U, delta_pi,
                     n_mc=N_MC, seed=MC_SEED, d_correlated=True, z_error=0.0):
    rows = []
    for J in J_values:
        rng = np.random.default_rng(seed)
        d = pd.DataFrame([study3_one_dataset(J, discordance_rate, pi_U, delta_pi,
                                             rng, N_BOOT_POWER, d_correlated, z_error)
                          for _ in range(n_mc)])
        rows.append({"J": J, "n_mc": n_mc, "discordance_rate": discordance_rate,
                     "pi_U": pi_U, "true_delta_pi": delta_pi,
                     "d_correlated": d_correlated, "z_error": z_error,
                     "mean_delta_hat": d.delta_hat.mean(), "power": d.reject.mean(),
                     "mean_D_discordant_pairs": d.n_D_discordant.mean(),
                     "mean_U_discordant_pairs": d.n_U_discordant.mean()})
        print(f"J={J:3d} | disc={discordance_rate:.2f} Δπ={delta_pi:.2f} "
              f"corr={d_correlated} zerr={z_error:.2f} | power={d.reject.mean():.3f} "
              f"| D-pairs≈{d.n_D_discordant.mean():.0f}")
    return pd.DataFrame(rows)

print("=== MAIN (corrected: correlated wins, 15% Z error) ===")
study3_main = run_study3_power(J_GRID, 0.25, 0.50, 0.10,
                               seed=MC_SEED + 50_000, d_correlated=True, z_error=0.15)

print("\n=== Decomposition of the two corrections at J=120 ===")
decomp = pd.concat([
    run_study3_power([120], 0.25, 0.50, 0.10, seed=MC_SEED+1, d_correlated=False, z_error=0.00),
    run_study3_power([120], 0.25, 0.50, 0.10, seed=MC_SEED+2, d_correlated=True,  z_error=0.00),
    run_study3_power([120], 0.25, 0.50, 0.10, seed=MC_SEED+3, d_correlated=False, z_error=0.15),
    run_study3_power([120], 0.25, 0.50, 0.10, seed=MC_SEED+4, d_correlated=True,  z_error=0.15),
], ignore_index=True)
display(decomp[["d_correlated","z_error","power","mean_delta_hat"]].round(3))

print("\n=== Sensitivity at J=120 (corrected model) ===")
sens = pd.concat([run_study3_power([120], dr, 0.50, dp,
                                   seed=MC_SEED + int(dr*1000) + int(dp*10000),
                                   d_correlated=True, z_error=0.15)
                  for dr in [0.15, 0.25, 0.35] for dp in [0.05, 0.10, 0.15]],
                 ignore_index=True)
display(sens[["discordance_rate","true_delta_pi","power",
              "mean_D_discordant_pairs","mean_U_discordant_pairs"]].round(3))

study3_main.to_csv(MC_DIR / "study3_power_main.csv", index=False)
sens.to_csv(MC_DIR / "study3_power_sensitivity_J120.csv", index=False)
decomp.to_csv(MC_DIR / "study3_power_decomposition_J120.csv", index=False)

=== MAIN (corrected: correlated wins, 15% Z error) ===
J= 80 | disc=0.25 Δπ=0.10 corr=True zerr=0.15 | power=0.158 | D-pairs≈83
J=100 | disc=0.25 Δπ=0.10 corr=True zerr=0.15 | power=0.159 | D-pairs≈104
J=120 | disc=0.25 Δπ=0.10 corr=True zerr=0.15 | power=0.182 | D-pairs≈125
J=150 | disc=0.25 Δπ=0.10 corr=True zerr=0.15 | power=0.227 | D-pairs≈157

=== Decomposition of the two corrections at J=120 ===
J=120 | disc=0.25 Δπ=0.10 corr=False zerr=0.00 | power=0.477 | D-pairs≈125
J=120 | disc=0.25 Δπ=0.10 corr=True zerr=0.00 | power=0.318 | D-pairs≈125
J=120 | disc=0.25 Δπ=0.10 corr=False zerr=0.15 | power=0.284 | D-pairs≈125
J=120 | disc=0.25 Δπ=0.10 corr=True zerr=0.15 | power=0.175 | D-pairs≈125


   d_correlated  z_error  power  mean_delta_hat
0         False     0.00  0.478           0.098
1          True     0.00  0.318           0.100
2         False     0.15  0.284           0.069
3          True     0.15  0.176           0.069



=== Sensitivity at J=120 (corrected model) ===
J=120 | disc=0.15 Δπ=0.05 corr=True zerr=0.15 | power=0.065 | D-pairs≈75
J=120 | disc=0.15 Δπ=0.10 corr=True zerr=0.15 | power=0.147 | D-pairs≈75
J=120 | disc=0.15 Δπ=0.15 corr=True zerr=0.15 | power=0.281 | D-pairs≈75
J=120 | disc=0.25 Δπ=0.05 corr=True zerr=0.15 | power=0.081 | D-pairs≈125
J=120 | disc=0.25 Δπ=0.10 corr=True zerr=0.15 | power=0.178 | D-pairs≈125
J=120 | disc=0.25 Δπ=0.15 corr=True zerr=0.15 | power=0.352 | D-pairs≈125
J=120 | disc=0.35 Δπ=0.05 corr=True zerr=0.15 | power=0.089 | D-pairs≈175
J=120 | disc=0.35 Δπ=0.10 corr=True zerr=0.15 | power=0.215 | D-pairs≈175
J=120 | disc=0.35 Δπ=0.15 corr=True zerr=0.15 | power=0.406 | D-pairs≈175


   discordance_rate  true_delta_pi  power  mean_D_discordant_pairs  \
0              0.15           0.05  0.065                   75.130   
1              0.15           0.10  0.148                   75.054   
2              0.15           0.15  0.280                   74.782   
3              0.25           0.05  0.081                  125.394   
4              0.25           0.10  0.178                  125.202   
5              0.25           0.15  0.352                  125.049   
6              0.35           0.05  0.090                  175.165   
7              0.35           0.10  0.215                  175.110   
8              0.35           0.15  0.406                  175.077   

   mean_U_discordant_pairs  
0                  214.886  
1                  215.544  
2                  215.736  
3                  359.684  
4                  358.959  
5                  359.762  
6                  502.950  
7                  503.566  
8                  502.098  


In [ ]:
# ============================================================
# 18. JOINT HUMAN-SAMPLE PLANNING SUMMARY
# ============================================================
# Study 1 is the formal power gate for J. Study 3's human-subset power is
# descriptive: its primary test runs on the LLM-scored corpus.

print("=" * 78); print("STUDY 1 MINIMUM FOCAL POWER, ALL SCENARIOS (one-sided Γ)")
print("=" * 78)
s1 = study1_power.copy()
s1["min_power"] = s1[["power_delta_C","power_delta_E","power_gamma_1sided"]].min(axis=1)
display(s1.pivot(index="J", columns="scenario", values="min_power").round(3))
print("Under `conservative`, no candidate J reaches .80 -- report this, do not")
print("present `planning` as if it were the only scenario examined.")

joint = (s1[s1.scenario == "planning"]
         [["J","power_delta_C","power_delta_E","power_gamma_1sided",
           "power_gamma_2sided","power_all_three","min_power","implied_human_alpha"]]
         .merge(study3_main[["J","power","mean_D_discordant_pairs",
                             "mean_U_discordant_pairs"]], on="J", how="left")
         .rename(columns={"power": "study3_human_power"}))

joint["responses"]     = joint.J * N_RESPONSES_PER_OP
joint["unique_pairs"]  = joint.J * N_PAIRS_PER_OP
joint["pair_assign"]   = joint.unique_pairs * N_HUMAN_RATERS
joint["construct_judg"]= joint.pair_assign * 2
joint["passes_80"]     = joint.min_power >= 0.80
joint["planned"]       = joint.J == 120

print("\n" + "=" * 78); print("HUMAN-SAMPLE PLANNING TABLE (planning scenario)")
print("=" * 78)
display(joint.round(3))

p = joint[joint.J == 120].iloc[0]
print(f"\nPLANNED J=120")
print(f"  Study 1 min focal power (1-sided Γ): {p.min_power:.3f}")
print(f"  All three survive Holm:              {p.power_all_three:.3f}")
print(f"  Implied human alpha:                 {p.implied_human_alpha:.2f}")
print(f"  Study 3 human-subset power:          {p.study3_human_power:.3f}  (descriptive)")
print(f"  Burden: {int(p.responses)} responses / {int(p.unique_pairs):,} pairs / "
      f"{int(p.pair_assign):,} assignments / {int(p.construct_judg):,} judgments")

j150 = joint[joint.J == 150].iloc[0]
print(f"\nJ=120 -> 150: +{int(j150.construct_judg - p.construct_judg):,} judgments, "
      f"Γ power +{j150.power_gamma_1sided - p.power_gamma_1sided:.3f}, "
      f"Study 3 +{j150.study3_human_power - p.study3_human_power:.3f}")

joint.to_csv(MC_DIR / "joint_human_sample_size_decision_v3.csv", index=False)
print("\nSaved joint_human_sample_size_decision_v3.csv")

STUDY 1 MINIMUM FOCAL POWER, ALL SCENARIOS (one-sided Γ)


scenario  conservative  favorable  planning  wachsmuth_like
J                                                          
80               0.310      0.995     0.842             0.0
100              0.376      0.999     0.919             0.0
120              0.459      1.000     0.952             0.0
150              0.569      1.000     0.984             0.0


Under `conservative`, no candidate J reaches .80 -- report this, do not
present `planning` as if it were the only scenario examined.

HUMAN-SAMPLE PLANNING TABLE (planning scenario)


     J  power_delta_C  power_delta_E  power_gamma_1sided  power_gamma_2sided  \
0   80          0.956          0.963               0.842               0.742   
1  100          0.984          0.987               0.919               0.854   
2  120          0.996          0.996               0.952               0.906   
3  150          1.000          0.998               0.984               0.966   

   power_all_three  min_power  implied_human_alpha  study3_human_power  \
0            0.800      0.842                0.332               0.158   
1            0.902      0.919                0.333               0.159   
2            0.946      0.952                0.336               0.182   
3            0.982      0.984                0.337               0.226   

   mean_D_discordant_pairs  mean_U_discordant_pairs  responses  unique_pairs  \
0                   83.430                  239.599        480          1200   
1                  104.087                  299.930        600      


PLANNED J=120
  Study 1 min focal power (1-sided Γ): 0.952
  All three survive Holm:              0.946
  Implied human alpha:                 0.34
  Study 3 human-subset power:          0.182  (descriptive)
  Burden: 720 responses / 1,800 pairs / 5,400 assignments / 10,800 judgments

J=120 -> 150: +2,700 judgments, Γ power +0.032, Study 3 +0.044

Saved joint_human_sample_size_decision_v3.csv


In [ ]:
# ============================================================
# 19. MONTE CARLO ERROR + WHAT TO READ
# ============================================================
def mc_se(power, n): return np.sqrt(power * (1 - power) / n)
print(f"N_MC = {N_MC}")
print(f"MC SE at power=.80:        {mc_se(0.80, N_MC):.4f}")
print(f"95% MC margin at power=.80: ±{1.96 * mc_se(0.80, N_MC):.4f}")

print("\n" + "=" * 78); print("READ THESE THREE THINGS"); print("=" * 78)
g = study1_power[study1_power.scenario == "wachsmuth_like"]
print(f"1. wachsmuth_like mean Γ = {g.mean_gamma.mean():+.3f}")
print("   If NEGATIVE: RQ1b's one-sided direction is wrong under strong human")
print("   C/E coupling. Restate as two-sided or as equivalence BEFORE freezing")
print(f"   the rubric. Two-sided Γ power there: {g.power_gamma_2sided.mean():.3f}")

a = study1_power[study1_power.scenario == "planning"].implied_human_alpha.mean()
print(f"\n2. Implied human alpha at human_choice_scale=1.20: {a:.2f}")
print("   Published alpha for these dimensions runs .33-.45, so this is realistic")
print("   to mildly pessimistic. Report it -- it converts an arbitrary tuning")
print("   constant into a calibrated assumption.")

print(f"\n3. Study 3 corrected power at J=120: "
      f"{study3_main[study3_main.J==120].power.iloc[0]:.3f}")
print("   Lower than the uncorrected 0.52. See the decomposition table for how")
print("   much comes from within-OP correlation vs. Z misclassification. This")
print("   strengthens the existing argument for testing Study 3 primarily on the")
print("   LLM-scored corpus, not the 720-response human subset.")

N_MC = 2000
MC SE at power=.80:        0.0089
95% MC margin at power=.80: ±0.0175

READ THESE THREE THINGS
1. wachsmuth_like mean Γ = -0.135
   If NEGATIVE: RQ1b's one-sided direction is wrong under strong human
   C/E coupling. Restate as two-sided or as equivalence BEFORE freezing
   the rubric. Two-sided Γ power there: 0.915

2. Implied human alpha at human_choice_scale=1.20: 0.33
   Published alpha for these dimensions runs .33-.45, so this is realistic
   to mildly pessimistic. Report it -- it converts an arbitrary tuning
   constant into a calibrated assumption.

3. Study 3 corrected power at J=120: 0.182
   Lower than the uncorrected 0.52. See the decomposition table for how
   much comes from within-OP correlation vs. Z misclassification. This
   strengthens the existing argument for testing Study 3 primarily on the
   LLM-scored corpus, not the 720-response human subset.


## 20. Files used by the rest of the repository

`data/annotation_input.csv` holds only the text shown to annotators and models. `data/outcomes.csv` holds the behavioral outcomes, timing, position and sampling probabilities, without usernames, raw text, URLs or timestamps. Both are rebuilt byte-for-byte from the sample file.

In [ ]:
SAMPLE_SHA   = "289944870720a75f18b80c473185d76f223590c342f2e920d792044f835dd192"   # recorded when the sample was drawn
OUTCOMES_SHA = "292cb43b9d93598aa92d719d3f82c0d0f3ce58e4ba4cf0cfd99d3efdebf8a00d"   # data/outcomes.csv as released

sample_csv = DERIVED_DIR / "human_validation_sample_v3.csv"
assert sha256_file(sample_csv) == SAMPLE_SHA, "The sample differs from the one used in the study"

s = pd.read_csv(sample_csv, dtype=str, keep_default_na=False).sort_values(["post_id", "response_id"]).reset_index(drop=True)
posts = sorted(s.post_id.unique())
s["sample_post_order"] = s.post_id.map({p: i for i, p in enumerate(posts)})
s["sample_response_order"] = s.groupby("post_id").cumcount()

s[["sample_post_order", "sample_response_order", "post_id", "response_id", "op_title",
   "op_text_clean", "response_text_clean"]].to_csv(REPO / "data" / "annotation_input.csv", index=False, encoding="utf-8-sig")

keep = ["post_id", "response_id", "op_stratum", "behavior_state", "behavior_state_star",
        "op_reply", "op_delta", "response_score", "community_score_percentile",
        "hours_since_op", "response_order", "n_eligible_direct_responses",
        "inclusion_stratum", "n_available_in_stratum", "inclusion_prob_within_op"]
s[keep].to_csv(REPO / "data" / "outcomes.csv", index=False)
print("outcomes.csv matches the released file:", sha256_file(REPO / "data" / "outcomes.csv") == OUTCOMES_SHA)